# Meta-learning for few-shot domain adaptation (MAML)

The domain-adaptation study, set up as a **fair** test of episodic meta-learning.

Both arms start from the SAME MobileNet whose backbone + LSTM were first trained on
UCF101 (standard supervised). The only difference is what happens next:
- **No meta** - keep that UCF101-trained init and few-shot adapt to YouTube.
- **MAML** - additionally meta-train over UCF101 N-way K-shot episodes, then few-shot
  adapt to YouTube.

Because both saw the same UCF101 data and use the same few-shot adaptation, any gap is
due to episodic meta-training, not to one arm seeing more data.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
!pip install -q yt-dlp

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Setup**

N-way matches the number of YouTube classes. K-shot is the labelled budget per class at meta-test.

In [ ]:
N_WAY    = len(cfg.YOUTUBE_CLIPS)   # 4
K_SHOT   = 5
K_QUERY  = 5
EVAL_DRAWS = 10   # random few-shot support draws at meta-test (both arms averaged over these)

PRETRAIN_EPOCHS = 5      # shared UCF101 supervised training (both arms)
META_EPOCHS     = 3      # extra episodic meta-training (MAML arm only)
META_EPISODES   = 40
INNER_LR        = 0.01
INNER_STEPS     = 5
META_LR         = 1e-3
print(f"{N_WAY}-way {K_SHOT}-shot | {EVAL_DRAWS} eval draws")

## **Preprocess data**

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [ ]:
# BASE  (max_samples caps TRAIN clips/class via cfg.MAX_SAMPLES; None = full data, val/test untouched)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 1  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 2  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}
print(f"{len(ALL_CLASSES_LIST)} UCF101 classes")

## **Create Loaders**

Combined UCF101 loaders for the shared supervised pretraining, and a RAM pool for the meta-training episodes.

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_val    = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "val"),   class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_val   = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "val"),   class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_val   = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "val"),   class_to_idx=class_to_idx)

train_all_loader = DataLoader(ConcatDataset([base_train, task1_train, task2_train]), batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
val_all_loader   = DataLoader(ConcatDataset([base_val,   task1_val,   task2_val]),   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
pool_clips, pool_labels = build_pool([base_train, task1_train, task2_train], per_class=20)
class_pool = sorted(set(pool_labels.tolist()))
print(f"pool {tuple(pool_clips.shape)}  classes {len(class_pool)}")

## **Shared backbone (UCF101 pretraining)**

Train one MobileNet on UCF101 (all classes). Both arms start from this - it is the fair, common starting point.

In [ ]:
set_seed(cfg.SEED)
pretrained = MobileNetV3SmallLSTMStudent(num_classes=len(ALL_CLASSES_LIST)).to(device)
hist_pre = train_model(pretrained, train_all_loader, val_all_loader, num_epochs=PRETRAIN_EPOCHS, lr=1e-3, device=device)

In [ ]:
plot_training_results(hist_pre, task_name="MobileNet on UCF101")

## **Common init**

Take the UCF101-trained backbone + LSTM and give it a fresh N-way head. Both arms copy this.

In [ ]:
import copy
init_model = copy.deepcopy(pretrained)
init_model.fc = nn.Linear(cfg.HIDDEN_SIZE, N_WAY).to(device)
print(init_model.fc)

## **YouTube few-shot task**

Download the real out-of-domain YouTube clips. We do NOT fix one support set here - the fairness
fix is to draw many random K-shot support sets below and average both arms over them.

In [ ]:
yt_clips, yt_labels = download_youtube_clips(cfg.YOUTUBE_CLIPS, class_to_idx, n_clips=40)
yt_ids = sorted(set(yt_labels.tolist()))
print(f"youtube clips {tuple(yt_clips.shape)}  classes {len(yt_ids)}")

## **MAML (first-order)**

Meta-train the shared init on UCF101 N-way K-shot episodes so it learns to adapt fast from few
clips. Only this arm gets meta-training; the no-meta arm keeps the plain pretrained init.

In [ ]:
set_seed(cfg.SEED)
maml_model = copy.deepcopy(init_model)
maml_model, hist_maml = meta_train_maml(maml_model, pool_clips, pool_labels, class_pool,
                                        N_WAY, K_SHOT, K_QUERY, META_EPOCHS, META_EPISODES,
                                        INNER_LR, INNER_STEPS, META_LR, device)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, META_EPOCHS + 1), hist_maml["query_accs"], marker="s", label="MAML")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training progress (UCF101 episodes)")
plt.legend(); plt.tight_layout(); plt.show()

## **Few-shot adaptation to YouTube (N random draws)**

The fair comparison. For each draw we sample the SAME random K-shot support + query and adapt
BOTH arms on it (same inner_lr, same inner_steps) - the only difference is the init (plain vs
meta-trained). A single 5-shot draw is noisy, so we repeat EVAL_DRAWS times and report mean +/- std.
If MAML's meta-trained init really helps it learn from few clips, it shows up as a higher mean here.

In [ ]:
set_seed(cfg.SEED)
base_accs, maml_accs = [], []
for d in range(EVAL_DRAWS):
    # same support/query for both arms this draw -> the only difference is the init
    sx, sy, qx, qy = make_task(yt_clips, yt_labels, yt_ids, K_SHOT, 1000, device)
    base_accs.append(few_shot_eval(copy.deepcopy(init_model), sx, sy, qx, qy, INNER_LR, INNER_STEPS))
    maml_accs.append(few_shot_eval(copy.deepcopy(maml_model), sx, sy, qx, qy, INNER_LR, INNER_STEPS))

In [ ]:
base_mean, base_std = float(np.mean(base_accs)), float(np.std(base_accs))
maml_mean, maml_std = float(np.mean(maml_accs)), float(np.std(maml_accs))
print(f"no-meta : {base_mean:.4f} +/- {base_std:.4f}")
print(f"MAML    : {maml_mean:.4f} +/- {maml_std:.4f}")
print(f"MAML gain over no-meta: {maml_mean - base_mean:+.4f}  (over {EVAL_DRAWS} draws)")

## **Compare**

In [ ]:
methods = ["no meta", "MAML"]
means   = [base_mean, maml_mean]
stds    = [base_std, maml_std]
plt.figure(figsize=(6, 5))
plt.bar(methods, means, yerr=stds, capsize=8)
plt.ylim(0, 1.05); plt.ylabel("YouTube few-shot accuracy")
plt.title(f"{N_WAY}-way {K_SHOT}-shot adaptation to YouTube (mean +/- std over {EVAL_DRAWS} draws)")
plt.tight_layout(); plt.show()

## **Save**

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"n_way": N_WAY, "k_shot": K_SHOT, "eval_draws": EVAL_DRAWS,
           "shared_init": "ucf101_pretrained",
           "no_meta_mean": base_mean, "no_meta_std": base_std,
           "maml_mean": maml_mean, "maml_std": maml_std,
           "no_meta_accs": base_accs, "maml_accs": maml_accs,
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/stage5_meta_domain_adapt.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/stage5_meta_domain_adapt.json")